# Ayanami0730/rag_test Dataset Exploration

Explore the structure of the A-RAG benchmark dataset cached at `~/.cache/huggingface/datasets--Ayanami0730--rag_test`.

**Datasets:** musique, hotpotqa, 2wikimultihop, medical, novel

Each dataset has:
- `chunks.json`: corpus in `"id:text"` format
- `questions.json`: evaluation questions with id, source, question, answer, evidence

In [1]:
from pathlib import Path
import json

HF_CACHE = Path("/projects/prjs1800/.cache/huggingface/datasets--Ayanami0730--rag_test")
SNAPSHOT = HF_CACHE / "snapshots" / "b9198a5a8702cc35c6df7542529357a9af95d928"
DATASETS = ["hotpotqa", "musique", "2wikimultihop", "medical", "novel"]

print("Cache exists:", HF_CACHE.exists())
print("Snapshot exists:", SNAPSHOT.exists())
print("Datasets:", [d for d in DATASETS if (SNAPSHOT / d).exists()])

Cache exists: True
Snapshot exists: True
Datasets: ['hotpotqa', 'musique', '2wikimultihop', 'medical', 'novel']


## Dataset structure (files per split)

In [ ]:
for ds in DATASETS:
    dpath = SNAPSHOT / ds
    if not dpath.exists():
        continue
    files = list(dpath.iterdir())
    print(f"{ds}:")
    for f in sorted(files):
        
        size = f.stat().st_size if f.is_file() else "(symlink)"
        print(f"  {f.name}: {size}")
    print()

hotpotqa:
  chunks.json: 6152182
  questions.json: 7088639

musique:
  chunks.json: 6390683
  questions.json: 278704

2wikimultihop:
  chunks.json: 2944188
  questions.json: 777574

medical:
  chunks.json: 1174306
  questions.json: 2176296

novel:
  chunks.json: 4858330
  questions.json: 1379466



## Chunks format (`id:text`)

In [3]:
def load_chunks(dataset: str) -> list:
    path = SNAPSHOT / dataset / "chunks.json"
    with open(path) as f:
        return json.load(f)

def parse_chunk(entry: str) -> tuple[str, str]:
    """Parse 'id:text' format. First colon separates id from text."""
    idx = entry.find(":")
    if idx < 0:
        return ("", entry)
    return (entry[:idx], entry[idx + 1:])

# Sample from hotpotqa
chunks = load_chunks("hotpotqa")
print(f"hotpotqa chunks: {len(chunks)} entries")
print(f"Type: {type(chunks[0])}")
cid, text = parse_chunk(chunks[0])
print(f"First chunk id: {cid!r}")
print(f"First chunk text (first 200 chars): {text[:200]!r}")

hotpotqa chunks: 1311 entries
Type: <class 'str'>
First chunk id: '0'
First chunk text (first 200 chars): 'vaada poda nanbargal is a 2011 indian tamil - language romantic comedy film directed by manikai. p. arumaichandran has produced this movie under the banner 8 point entertainments. the film stars newco'


## Questions format

In [5]:
def load_questions(dataset: str) -> list:
    path = SNAPSHOT / dataset / "questions.json"
    with open(path) as f:
        return json.load(f)

q = load_questions("hotpotqa")
print(f"hotpotqa questions: {len(q)}")
print("Keys:", list(q[0].keys()))
print("Sample:")
for k, v in q[0].items():
    val = str(v)[:80] + "..." if len(str(v)) > 80 else v
    print(f"  {k}: {val}")

hotpotqa questions: 1000
Keys: ['id', 'source', 'question', 'answer', 'question_type', 'evidence']
Sample:
  id: 5abe953b5542993f32c2a170
  source: hotpotqa
  question: what is one of the stars of  The Newcomers known for
  answer: superhero roles as the Marvel Comics
  question_type: bridge
  evidence: [['Vaada Poda Nanbargal', ['Vaada Poda Nanbargal is a 2011 Indian Tamil-language...


## Summary: chunks & questions per dataset

In [7]:
rows = []
for ds in DATASETS:
    try:
        c = load_chunks(ds)
        q = load_questions(ds)
        rows.append((ds, len(c), len(q)))
    except FileNotFoundError:
        rows.append((ds, "—", "—"))

print(f"{'Dataset':<15} {'Chunks':>8} {'Questions':>10}")
print("-" * 35)
for ds, nc, nq in rows:
    print(f"{ds:<15} {nc:>8} {nq:>10}")

Dataset           Chunks  Questions
-----------------------------------
hotpotqa            1311       1000
musique             1354       1000
2wikimultihop        658       1000
medical              225       2062
novel               1117       2010


## Sample questions by dataset

In [8]:
for ds in ["hotpotqa", "musique", "2wikimultihop"]:
    q = load_questions(ds)
    ex = q[0]
    print(f"--- {ds} ---")
    print(f"id: {ex.get('id')}")
    print(f"question: {ex.get('question', '')[:100]}...")
    print(f"answer: {ex.get('answer')}")
    print(f"question_type: {ex.get('question_type')}")
    print()

--- hotpotqa ---
id: 5abe953b5542993f32c2a170
question: what is one of the stars of  The Newcomers known for...
answer: superhero roles as the Marvel Comics
question_type: bridge

--- musique ---
id: musique_2hop__13548_13529
question: When was the person who Messi's goals in Copa del Rey compared to get signed by Barcelona?...
answer: June 1982
question_type: 

--- 2wikimultihop ---
id: 83bf3b5a0bd911eba7f7acde48001122
question: When did Lothair Ii's mother die?...
answer: 20 March 851
question_type: compositional



## Chunk length distribution (hotpotqa)

In [9]:
chunks = load_chunks("hotpotqa")
lengths = [len(parse_chunk(c)[1]) for c in chunks]
print(f"Min: {min(lengths)}, Max: {max(lengths)}, Mean: {sum(lengths)/len(lengths):.0f}")

# Simple histogram
bins = [0, 100, 200, 500, 1000, 2000, 10000]
for i in range(len(bins) - 1):
    n = sum(1 for L in lengths if bins[i] <= L < bins[i+1])
    print(f"  [{bins[i]:>5}, {bins[i+1]:>5}): {n}")

Min: 3744, Max: 5388, Mean: 4646
  [    0,   100): 0
  [  100,   200): 0
  [  200,   500): 0
  [  500,  1000): 0
  [ 1000,  2000): 0
  [ 2000, 10000): 1311


## Zly0523/linear-rag: Download & Compare

A-RAG/rag_test reformats from LinearRAG. Verify hotpotqa, musique, 2wikimultihop, medical are identical.

In [10]:
from huggingface_hub import snapshot_download

LINEAR_RAG_DIR = Path("/projects/prjs1800/.cache/huggingface/linear-rag-download")
if not (LINEAR_RAG_DIR / "hotpotqa" / "chunks.json").exists():
    snapshot_download(repo_id="Zly0523/linear-rag", repo_type="dataset", local_dir=str(LINEAR_RAG_DIR))
    print("Downloaded Zly0523/linear-rag")
else:
    print("Linear-rag already at:", LINEAR_RAG_DIR)

/projects/prjs1800/venvs/arag-venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Linear-rag already at: /projects/prjs1800/.cache/huggingface/linear-rag-download


In [11]:
# Compare rag_test vs linear-rag (overlapping datasets)
OVERLAP = ["hotpotqa", "musique", "2wikimultihop", "medical"]

def load_pair(dataset: str):
    rt_c = json.load(open(SNAPSHOT / dataset / "chunks.json"))
    rt_q = json.load(open(SNAPSHOT / dataset / "questions.json"))
    lr_c = json.load(open(LINEAR_RAG_DIR / dataset / "chunks.json"))
    lr_q = json.load(open(LINEAR_RAG_DIR / dataset / "questions.json"))
    return (rt_c, rt_q), (lr_c, lr_q)

print(f"{'Dataset':<15} {'Chunks':>8} {'Questions':>10} {'Chunks_eq':>10} {'Q_ids_eq':>10}")
print("-" * 55)
for ds in OVERLAP:
    (rt_c, rt_q), (lr_c, lr_q) = load_pair(ds)
    chunks_eq = rt_c == lr_c
    q_ids_eq = {x["id"] for x in rt_q} == {x["id"] for x in lr_q}
    print(f"{ds:<15} {len(rt_c):>8} {len(rt_q):>10} {str(chunks_eq):>10} {str(q_ids_eq):>10}")

print("\nrag_test == linear-rag for hotpotqa, musique, 2wikimultihop, medical.")
print("rag_test adds 'novel' from GraphRAG-Bench (not in linear-rag).")

Dataset           Chunks  Questions  Chunks_eq   Q_ids_eq
-------------------------------------------------------
hotpotqa            1311       1000       True       True
musique             1354       1000       True       True
2wikimultihop        658       1000       True       True
medical              225       2062       True       True

rag_test == linear-rag for hotpotqa, musique, 2wikimultihop, medical.
rag_test adds 'novel' from GraphRAG-Bench (not in linear-rag).
